This notebook contains the probability generation function with decision estimation using the finally selected probability thresholds.

We convert the predicted urgency labels to user-facing actions

# Action Estimation

In [ ]:
import joblib

df = joblib.load("dnd_df.pkl")
model = joblib.load("dnd_rf_model.pkl")
vectorizer = joblib.load("dnd_vectorizer.pkl")
metadata_features = joblib.load("metadata_features.pkl")

In [ ]:
from scipy.sparse import csr_matrix, hstack

def score_message(
    message,
    sender_importance,
    is_group_chat,
    hour_of_day,
    average_messages_per_day
):

    X_text = vectorizer.transform([message])

    X_meta = csr_matrix([[
        sender_importance,
        is_group_chat,
        hour_of_day,
        average_messages_per_day
    ]])

    X_combined = hstack([X_text, X_meta])

    probability = model.predict_proba(X_combined)[0][1]

    return probability

In [ ]:
def notification_action(prob):

    if prob >= 0.65:
        return "BYPASS_DND"

    elif prob >= 0.40:
        return "SILENT_SUMMARY"

    else:
        return "SUPPRESS"

# Actions

In [ ]:
# placeholder functions for actions
def send_notification(message):
    return {
        "action": "BYPASS_DND",
        "result": f"Notification delivered: {message}"
    }

def add_to_summary(message):
    return {
        "action": "SILENT_SUMMARY",
        "result": f"Added to summary: {message}"
    }

def suppress_notification(message):
    return {
        "action": "SUPPRESS",
        "result": f"Notification suppressed"
    }

In [ ]:
# process incoming message
def process_notification(
    message,
    sender_importance,
    is_group_chat,
    hour_of_day,
    average_messages_per_day
):

    probability = score_message(
        message,
        sender_importance,
        is_group_chat,
        hour_of_day,
        average_messages_per_day
    )

    decision = notification_action(probability)

    if decision == "BYPASS_DND":
        action_result = send_notification(message)

    elif decision == "SILENT_SUMMARY":
        action_result = add_to_summary(message)

    else:
        action_result = suppress_notification(message)

    return {
        "message": message,
        "probability": round(probability, 3),
        "decision": decision,
        "action_result": action_result
    }

In [ ]:
# test
process_notification(
    message="Please call me immediately",
    sender_importance=5,
    is_group_chat=0,
    hour_of_day=23,
    average_messages_per_day=1
)

In [ ]:
# test
# test
import pandas as pd

test_messages = [
    {
        "message": "Please call me immediately",
        "sender_importance": 5,
        "is_group_chat": 0,
        "hour_of_day": 23,
        "average_messages_per_day": 1
    },
    {
        "message": "Dad is in the hospital",
        "sender_importance": 5,
        "is_group_chat": 0,
        "hour_of_day": 1,
        "average_messages_per_day": 1
    },
    {
        "message": "Can you review this tomorrow?",
        "sender_importance": 3,
        "is_group_chat": 0,
        "hour_of_day": 14,
        "average_messages_per_day": 2
    },
    {
        "message": "Team lunch at 12?",
        "sender_importance": 2,
        "is_group_chat": 1,
        "hour_of_day": 10,
        "average_messages_per_day": 15
    },
    {
        "message": "LOL that's hilarious",
        "sender_importance": 2,
        "is_group_chat": 1,
        "hour_of_day": 20,
        "average_messages_per_day": 40
    },
    {
        "message": "The server is down, please check ASAP",
        "sender_importance": 5,
        "is_group_chat": 0,
        "hour_of_day": 2,
        "average_messages_per_day": 1
    },
    {
        "message": "Don't forget to bring snacks tomorrow",
        "sender_importance": 2,
        "is_group_chat": 1,
        "hour_of_day": 18,
        "average_messages_per_day": 10
    },
    {
        "message": "Emergency at home, call me now",
        "sender_importance": 5,
        "is_group_chat": 0,
        "hour_of_day": 3,
        "average_messages_per_day": 1
    },
    {
        "message": "Meeting moved to 3 PM",
        "sender_importance": 3,
        "is_group_chat": 0,
        "hour_of_day": 9,
        "average_messages_per_day": 3
    },
    {
        "message": "Are you free this weekend?",
        "sender_importance": 2,
        "is_group_chat": 0,
        "hour_of_day": 17,
        "average_messages_per_day": 2
    },
    {
        "message":"Can you call me when you get a chance?",
        "sender_importance":5,
        "is_group_chat":0,
        "hour_of_day":22,
        "average_messages_per_day":2
    }
]

results = []

for msg in test_messages:
    output = process_notification(**msg)

    results.append({
        "Message": output["message"],
        "Probability": round(float(output["probability"]), 3),
        "Decision": output["decision"],
        "Action": output["action_result"]["result"]
    })

simulation_df = pd.DataFrame(results)

simulation_df.sort_values(
    by="Probability",
    ascending=False
).reset_index(drop=True)